# Tempahan Hotel dengan Middleware Ahli Keutamaan

Notebook ini menunjukkan **middleware berasaskan fungsi** menggunakan Microsoft Agent Framework. Kami membina berdasarkan contoh aliran kerja bersyarat dengan menambahkan lapisan middleware yang memberikan keistimewaan khas kepada ahli keutamaan.

## Apa yang Anda Akan Pelajari:
1. **Middleware Berasaskan Fungsi**: Mencegat dan mengubah keputusan fungsi
2. **Akses Konteks**: Membaca dan mengubah `context.result` selepas pelaksanaan
3. **Pelaksanaan Logik Perniagaan**: Manfaat ahli keutamaan
4. **Gantian Keputusan**: Menukar hasil fungsi berdasarkan status pengguna
5. **Aliran Kerja Sama, Hasil Berbeza**: Perubahan tingkah laku dipengaruhi middleware

## Seni Bina Aliran Kerja dengan Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Perbezaan Utama daripada Aliran Kerja Bersyarat:

**Tanpa Middleware** (14-conditional-workflow.ipynb):
- Paris tiada bilik → Laluan ke alternative_agent

**Dengan Middleware** (notebook ini):
- Pengguna biasa + Paris → Tiada bilik → Laluan ke alternative_agent
- Pengguna keutamaan + Paris → 🌟 Middleware mengekalkan! → Ada bilik → Laluan ke booking_agent

## Prasyarat:
- Microsoft Agent Framework dipasang
- Memahami aliran kerja bersyarat (rujuk 14-conditional-workflow.ipynb)
- Token GitHub atau kekunci API OpenAI
- Pemahaman asas corak middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Langkah 1: Takrifkan Model Pydantic untuk Output Berstruktur

Model-model ini mentakrifkan **skema** yang akan dipulangkan oleh ejen. Kami telah menambah medan `priority_override` untuk mengesan bila middleware mengubah keputusan ketersediaan.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Langkah 2: Takrifkan Pangkalan Data Ahli Keutamaan

Untuk demo ini, kami akan mensimulasikan pangkalan data keahlian keutamaan. Dalam pengeluaran, ini akan memanggil pangkalan data sebenar atau API.

**Ahli Keutamaan:**
- `alice@example.com` - ahli VIP
- `bob@example.com` - ahli Premium  
- `priority_user` - akaun ujian


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Langkah 3: Buat Alat Tempahan Hotel

Sama seperti aliran kerja bersyarat, tetapi kini ia akan disekat oleh middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 Cipta Middleware Semakan Keutamaan (CIRI UTAMA!)

Ini adalah **fungsi teras** bagi buku nota ini. Middleware:

1. **Menyekat** panggilan fungsi hotel_booking
2. **Melaksanakan** fungsi seperti biasa dengan memanggil `next(context)`
3. **Memeriksa** keputusan dalam `context.result`
4. **Menetapkan semula** keputusan jika pengguna adalah keutamaan dan tiada bilik tersedia
5. **Mengembalikan** keputusan yang diubah balik kepada agen

**Corak Utama:**
```python
async def my_middleware(context, next):
    await next(context)  # Laksanakan fungsi
    # Sekarang context.result mengandungi output fungsi
    if some_condition:
        context.result = new_value  # Timpa!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Langkah 5: Definisikan Fungsi Syarat untuk Penentuan Laluan

Fungsi syarat yang sama seperti aliran kerja bersyarat - mereka memeriksa output berstruktur untuk menentukan laluan.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Langkah 6: Buat Pelaksana Paparan Tersuai

Pelaksana yang sama seperti sebelum ini - memaparkan output aliran kerja akhir.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Langkah 7: Muatkan Pembolehubah Persekitaran

Konfigurasikan klien LLM (Model GitHub atau OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Langkah 8: Cipta Ejen AI dengan Middleware

**PERBEZAAN UTAMA:** Apabila mencipta availability_agent, kami memasukkan parameter `middleware`!

Begini cara kami menyuntik priority_check_middleware ke dalam saluran pelaksanaan fungsi ejen.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Langkah 9: Bina Aliran Kerja

Struktur aliran kerja yang sama seperti sebelum ini - penghalaan bersyarat berdasarkan ketersediaan.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: Kes Ujian 1 - Pengguna Biasa di Paris (Tiada Gantian)

Seorang pengguna biasa cuba menempah Paris → Tiada bilik → Menghala ke alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Langkah 11: Kes Ujian 2 - 🌟 Pengguna Keutamaan di Paris (DENGAN Gantian!)

Seorang ahli keutamaan cuba membuat tempahan Paris → Tiada bilik pada mulanya → 🌟 Middleware menggantikan! → Menghala ke booking_agent

**Ini adalah demonstrasi utama kuasa middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Langkah 12: Kes Ujian 3 - Pengguna Keutamaan di Stockholm (Sudah Tersedia)

Pengguna keutamaan cuba Stockholm → Bilik tersedia → Tiada penggantian diperlukan → Menghala ke booking_agent

Ini menunjukkan bahawa middleware hanya bertindak apabila diperlukan!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Pengajaran Utama dan Konsep Middleware

### ✅ Apa yang Anda Pelajari:

#### **1. Corak Middleware Berasaskan Fungsi**

Middleware menyekat panggilan fungsi menggunakan fungsi async yang ringkas:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Sebelum pelaksanaan fungsi
    print("Intercepting...")
    
    # Laksanakan fungsi
    await next(context)
    
    # Selepas pelaksanaan fungsi - periksa keputusan
    if context.result:
        # Ubah suai keputusan jika perlu
        context.result = modified_value
```

#### **2. Akses Konteks dan Gantian Keputusan**

- `context.function` - Akses fungsi yang dipanggil
- `context.arguments` - Baca argumen fungsi
- `context.kwargs` - Akses parameter tambahan
- `await next(context)` - Laksanakan fungsi
- `context.result` - Baca/ubah output fungsi

#### **3. Pelaksanaan Logik Perniagaan**

Middleware kami melaksanakan kelebihan ahli keutamaan:
- **Pengguna biasa**: Tiada pengubahsuaian, aliran kerja standard
- **Pengguna keutamaan**: Gantikan "tiada ketersediaan" → "tersedia"
- **Logik bersyarat**: Hanya gantikan apabila perlu

#### **4. Aliran Kerja Sama, Hasil Berbeza**

Kuasa middleware:
- ✅ Tiada perubahan pada struktur aliran kerja
- ✅ Tiada perubahan pada fungsi alat
- ✅ Tiada perubahan pada logik laluan bersyarat
- ✅ Hanya middleware → Tingkah laku berbeza!

### 🚀 Aplikasi Dunia Nyata:

1. **Ciri VIP/Premium**
   - Gantikan had kadar untuk pengguna premium
   - Berikan akses keutamaan ke sumber
   - Buka ciri premium secara dinamik

2. **Ujian A/B**
   - Mengarahkan pengguna ke pelaksanaan yang berbeza
   - Uji ciri baru dengan pengguna tertentu
   - Pelancaran ciri secara berkala

3. **Keselamatan & Pematuhan**
   - Audit panggilan fungsi
   - Sekat operasi sensitif
   - Memaksa peraturan perniagaan

4. **Pengoptimuman Prestasi**
   - Cache keputusan untuk pengguna tertentu
   - Langkau operasi mahal apabila boleh
   - Peruntukan sumber secara dinamik

5. **Pengendalian Ralat & Cuba Semula**
   - Tangkap dan kendalikan ralat dengan baik
   - Laksanakan logik cuba semula
   - Gunakan pelaksanaan alternatif

6. **Log & Pemantauan**
   - Jejak masa pelaksanaan fungsi
   - Log parameter dan keputusan
   - Pantau corak penggunaan

### 🔑 Perbezaan Utama dari Decorators:

| Ciri | Decorator | Middleware |
|---------|-----------|------------|
| **Skop** | Fungsi tunggal | Semua fungsi dalam agent |
| **Fleksibiliti** | Tetap semasa definisi | Dinamik semasa runtime |
| **Konteks** | Terhad | Konteks penuh agent |
| **Komposisi** | Pelbagai decorator | Aliran middleware |
| **Sedar Agent** | Tidak | Ya (akses keadaan agent) |

### 📚 Bila Nak Guna Middleware:

✅ **Gunakan middleware apabila:**
- Anda perlu ubah tingkah laku berdasarkan keadaan pengguna/sesi
- Anda ingin gunakan logik ke atas pelbagai fungsi
- Anda perlu akses kepada konteks tahap agent
- Anda melaksanakan keperluan merentas (logging, auth, dsb.)

❌ **Jangan guna middleware apabila:**
- Pengesahan input mudah (guna Pydantic)
- Logik khusus fungsi (simpankan dalam fungsi)
- Pengubahsuaian satu kali (ubah saja fungsi)

### 🎓 Corak Lanjutan:

```python
# Pelbagai middleware (susunan pelaksanaan penting!)
middleware=[
    logging_middleware,      # Log dahulu
    auth_middleware,         # Kemudian periksa pengesahan
    cache_middleware,        # Kemudian periksa cache
    rate_limit_middleware,   # Kemudian had kadar
    priority_check_middleware  # Akhirnya semakan keutamaan
]

# Pelaksanaan middleware bersyarat
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Ubah suai hasil
    else:
        # Langkau pelaksanaan sepenuhnya
        context.result = cached_value
```

### 🔗 Konsep Berkaitan:

- **Agent Middleware**: Menyekat panggilan agent.run()
- **Function Middleware**: Menyekat panggilan fungsi alat (yang kita guna!)
- **Middleware Pipeline**: Rantai middleware yang dilaksanakan mengikut urutan
- **Propagasi Konteks**: Mewariskan keadaan melalui rantai middleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
